In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col ,concat, lit , isnan, when, count
import pyspark.sql.functions as F

IMPORT BOTH DATASETS 

In [0]:
data = "/Volumes/workspace/default/my_files/laliga202425.csv"
data2 = "/Volumes/workspace/default/my_files/goalkeeping_laliga_2425.csv"
df = spark.read.csv(data, header=True, inferSchema=True)
display(df)

team,team_id,player_id,player_name,position,known_name,first_name,last_name,shirtNumber,played_time,appearances,played_games,starts,sub_on,sub_off,offsides,dribbles_successful,dribbles_unsuccessful,keypasses,assists,assists_intentional,assists_second_goal,through_balls,goals_conceded,goals_conceded_penalty,goals_conceded_outside_box,goals_conceded_inside_box,goals,goals_from_inside_box,goals_from_outside_box,goals_right_foot,goals_left_foot,goals_headed,goals_penalty,penalty_taken,shots_total,shots_on_target_include_goals,shots_off_target_include_woodwork,shots_hit_woodwork,shots_blocked,shots_attempts_from_set_pieces,passes_total,passes_total_exc_crosses_and_corners_succesful,passes_total_exc_crosses_and_corners_unsuccesful,passes_open_play,passes_open_play_successful,passes_backward,passes_forward,passes_leftside,passes_rightside,passes_short_successful,passes_short_unsuccessful,passes_long_successful,passes_long_unsuccessful,passes_own_half_successful,passes_own_half_unsuccessful,passes_opposition_half_successful,passes_opposition_half_unsuccessful,crosses_open_play_successful,crosses_open_play_unsuccessful,crosses_and_corners_unsuccessful,crosses_and_corners_successful,corners_won,corners_taken_include_short_corners,corners_into_box_successful,corners_into_box_unsuccessful,duels_total,duels_won,duels_lost,duels_ground_total,duels_ground_won,duels_ground_lost,duels_aerial_total,duels_aerial_won,duels_aerial_lost,tackles_total,tackles_won,tackles_lost,tackles_last_man,clearances,interceptions,recoveries,blocks
Getafe,1450,563206,Nabil Aberdin,Defender,null,Nabil,Aberdin,27,263,7,7,4,3,3,0,2,1,1,0,0,0,0,2,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,1,0,60,36,24,58,34,5,24,13,18,29,16,7,8,13,7,23,17,0,0,0,0,1,0,0,0,36,24,12,23,14,9,13,10,3,6,4,2,0,5,0,8,0
Getafe,1450,201895,Omar Alderete,Defender,null,Omar,Alderete,15,2978,34,34,34,0,7,8,18,10,11,0,0,0,3,35,8,6,29,1,1,0,0,0,1,0,0,27,10,17,1,11,7,1087,722,365,1020,677,46,576,130,335,574,150,148,215,391,77,333,292,2,4,4,2,9,0,0,0,286,174,112,130,71,59,156,103,53,43,29,14,0,238,40,115,22
Getafe,1450,179271,Carles Aleñá,Midfielder,null,Carles,Aleñá,11,1546,24,24,18,6,13,2,15,6,20,2,2,0,6,16,4,3,13,0,0,0,0,0,0,0,0,13,5,8,1,7,1,549,438,111,535,425,98,164,136,151,422,99,16,12,115,25,334,112,4,16,26,11,10,12,5,6,172,94,78,137,74,63,35,20,15,25,15,10,0,9,2,59,1
Getafe,1450,171222,Mauro Arambarri,Midfielder,null,Mauro,Arambarri,8,2664,35,35,31,4,8,3,18,12,17,0,0,0,2,32,5,6,26,10,9,1,6,2,2,3,3,42,19,23,3,14,1,782,584,198,765,567,109,255,205,213,551,131,33,67,239,47,347,163,2,8,12,2,7,1,0,1,347,177,170,232,118,114,115,59,56,48,29,19,0,43,15,137,6
Getafe,1450,514190,David Argüelles,Defender,null,David,Argüelles,34,6,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,3,2,5,3,2,0,2,1,3,1,0,1,0,0,3,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Getafe,1450,664920,Ismael Bekhoucha,Defender,null,Ismael,Bekhoucha,28,65,5,5,0,5,0,0,2,1,0,0,0,0,0,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,20,15,5,18,13,6,8,4,2,15,2,0,3,7,0,8,6,0,1,1,0,0,0,0,0,13,7,6,10,6,4,3,1,2,1,1,0,0,2,2,6,1
Getafe,1450,105046,Juan Bernat,Defender,null,Juan,Bernat,14,907,24,24,10,14,8,2,15,16,9,2,2,0,0,15,4,2,13,0,0,0,0,0,0,0,0,2,1,1,0,4,0,274,210,64,272,208,88,71,18,97,209,59,1,5,86,17,130,63,6,14,16,6,7,2,0,2,131,67,64,99,51,48,32,16,16,28,20,8,0,15,3,39,4
Getafe,1450,433632,Juan Berrocal,Defender,null,Juan,Berrocal,4,1060,19,19,11,8,5,0,0,0,1,0,0,0,0,12,2,2,10,0,0,0,0,0,0,0,0,3,0,3,0,1,0,377,287,90,358,271,17,162,110,88,261,39,26,51,196,19,91,71,0,0,0,0,0,0,0,0,83,45,38,35,15,20,48,30,18,14,7,7,0,58,15,35,3
Getafe,1450,149838,Djené Dakonam,Defender,null,Djené,Dakonam,2,2553,31,31,31,0,7,1,7,7,9,0,0,0,0,24,4,3,21,0,0,0,0,0,0,0,0,6,0,6,0,0,0,794,619,175,777,603,91,338,245,120,565,107,54,68,313,58,313,142,7,25,25,7,3,0,0,0,212,101,111,138,57,81,74,44,30,25,13,12,0,102,21,102,8
Getafe,1450,493338,Peter Federico González Carmona,Forward,Peter Federico,Peter Federico,González Carmona,19,467,17,17,2,15,2,2,3,10,4,0,0,0,0,6,3,0,6,0,0,0

In [0]:
gks2 = spark.read.csv(data2, header=True, inferSchema=True)
gks2 = gks2.drop("Rk", "Pos", "Squad", "Born", "MP", "Starts", "Min", "Matches", "-9999")
gks2 = gks2.withColumn("Nation", F.split(col("Nation"), " ").getItem(1))
gks2 = gks2.withColumnsRenamed({"Save%15": "Save%", "Save%25": "PK_Save%", "PKm": "PK_missed", "PKA": "PK_Allowed", "PKsv": "PK_Saved"})
gks2 = gks2.fillna({"PK_Save%": 0})
display(gks2)

Player,Nation,Age,90s,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt,PK_Allowed,PK_Saved,PK_missed,PK_Save%
Adrián,ESP,37,19.0,27,1.42,77,50,64.9,10,5,4,3,15.8,1,1,0,0,0.0
Julen Agirrezabala,ESP,23,13.4,10,0.75,34,24,70.6,9,4,1,5,35.7,2,1,1,0,50.0
Augusto Batalla,ARG,28,32.0,39,1.22,125,93,68.8,10,12,10,8,25.0,8,6,2,0,25.0
Dani Cárdenas,ESP,27,6.0,6,1.0,20,14,70.0,3,1,2,2,33.3,0,0,0,0,0.0
Jasper Cillessen,NED,35,25.9,42,1.62,130,88,67.7,6,8,13,3,11.1,8,4,3,1,42.9
Diego Conde,ESP,25,21.8,33,1.51,96,63,65.6,11,6,5,2,9.1,4,3,0,1,0.0
Thibaut Courtois,BEL,32,30.0,29,0.97,106,77,72.6,20,6,4,11,36.7,3,3,0,0,0.0
Stole Dimitrievski,MKD,30,4.0,6,1.5,12,6,50.0,0,2,2,0,0.0,2,1,0,1,0.0
Marko Dmitrovic,SRB,32,32.0,49,1.53,145,102,66.2,8,11,13,10,31.3,8,7,0,1,0.0
Aitor Fernández,ESP,33,1.0,0,0.0,5,5,100.0,0,1,0,1,100.0,0,0,0,0,0.0


In [0]:
gks = df.filter(df.position == 'Goalkeeper')
gks = gks.withColumn("player_name", when(col("known_name").isNotNull(), gks["known_name"]).otherwise(gks["player_name"]))
gks = gks.select('player_name', 'known_name', 'team', 'played_time', 'goals_conceded', 'goals_conceded_penalty', 'goals_conceded_outside_box', 'goals_conceded_inside_box', 'passes_total', 'passes_open_play', 'passes_open_play_successful', 'passes_forward', 'passes_leftside', 'passes_rightside', 'passes_short_successful', 'passes_short_unsuccessful', 'passes_long_successful', 'passes_long_unsuccessful', 'passes_own_half_successful', 'passes_own_half_unsuccessful', 'passes_opposition_half_successful', 'passes_opposition_half_unsuccessful', 'clearances', 'recoveries')

JOIN DE AMBOS DFs

In [0]:
gk_df = gks.join(gks2, gks.player_name == gks2.Player, how='full')
gk_df = gk_df.drop("Player")

#COMBINE KNOWN_NAME IF EXISTS
gk_df = gk_df.withColumn("player_name", when(col("known_name").isNotNull(), gk_df["known_name"]).otherwise(gk_df["player_name"]))

gk_df = gk_df.drop("known_name")


In [0]:
gk_df = gk_df.fillna(0)
gk_df.count()

45

In [0]:
gk_df.display()

player_name,team,played_time,goals_conceded,goals_conceded_penalty,goals_conceded_outside_box,goals_conceded_inside_box,passes_total,passes_open_play,passes_open_play_successful,passes_forward,passes_leftside,passes_rightside,passes_short_successful,passes_short_unsuccessful,passes_long_successful,passes_long_unsuccessful,passes_own_half_successful,passes_own_half_unsuccessful,passes_opposition_half_successful,passes_opposition_half_unsuccessful,clearances,recoveries,Nation,Age,90s,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt,PK_Allowed,PK_Saved,PK_missed,PK_Save%
Vicente Guaita,Celta de Vigo,3060,48,3,4,44,775,461,385,374,227,174,508,5,96,166,562,49,42,122,42,312,ESP,37,34.0,48,1.41,129,81,62.8,14,7,13,8,23.5,3,3,0,0,0.0
Augusto Batalla,Rayo Vallecano,2880,39,6,8,31,1044,676,486,681,177,185,447,5,212,380,543,53,116,332,26,286,ARG,28,32.0,39,1.22,125,93,68.8,10,12,10,8,25.0,8,6,2,0,25.0
Rui Silva,Real Betis,1350,18,3,0,18,452,320,264,193,117,142,285,3,57,107,312,23,30,87,8,129,POR,30,15.0,18,1.2,55,37,67.3,5,5,5,4,26.7,3,3,0,0,0.0
Pau López,Girona,45,2,0,0,2,22,13,8,15,0,7,12,0,4,6,12,1,4,5,1,5,ESP,29,0.5,2,4.0,3,1,33.3,0,0,0,0,0.0,0,0,0,0,0.0
Álex Padilla,Athletic Club,320,6,1,2,4,105,57,33,87,8,10,23,0,29,53,27,8,25,45,3,34,MEX,20,3.6,6,1.68,18,12,66.7,0,1,2,0,0.0,1,1,0,0,0.0
André Ferreira,Real Valladolid,540,20,3,0,20,168,84,61,104,34,30,75,0,37,56,86,15,26,41,9,43,POR,28,6.0,20,3.33,40,20,50.0,0,0,6,0,0.0,3,3,0,0,0.0
Marc-André ter Stegen,Barcelona,675,9,1,0,9,211,172,158,96,63,52,166,2,27,16,177,7,16,11,5,59,GER,32,7.5,9,1.2,23,14,60.9,7,0,1,1,12.5,1,1,0,0,0.0
Unai Marrero,Real Sociedad,180,3,0,0,3,45,26,24,18,18,9,26,1,9,9,32,2,3,8,2,12,ESP,22,2.0,3,1.5,11,8,72.7,0,0,2,0,0.0,1,0,1,0,100.0
Marko Dmitrovic,Leganés,2880,49,7,6,43,933,405,205,825,27,81,151,0,207,575,179,30,179,545,48,312,SRB,32,32.0,49,1.53,145,102,66.2,8,11,13,10,31.3,8,7,0,1,0.0
Julen Agirrezabala,Athletic Club,1207,9,1,2,7,356,232,175,225,51,80,162,0,69,125,183,20,48,105,11,105,ESP,23,13.4,10,0.75,34,24,70.6,9,4,1,5,35.7,2,1,1,0,50.0


In [0]:
gk_df.describe().display()

summary,player_name,team,played_time,goals_conceded,goals_conceded_penalty,goals_conceded_outside_box,goals_conceded_inside_box,passes_total,passes_open_play,passes_open_play_successful,passes_forward,passes_leftside,passes_rightside,passes_short_successful,passes_short_unsuccessful,passes_long_successful,passes_long_unsuccessful,passes_own_half_successful,passes_own_half_unsuccessful,passes_opposition_half_successful,passes_opposition_half_unsuccessful,clearances,recoveries,Nation,Age,90s,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt,PK_Allowed,PK_Saved,PK_missed,PK_Save%
count,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45,45
mean,null,null,1519.911111111111,22.11111111111111,2.3555555555555556,2.933333333333333,19.177777777777777,483.2,299.9111111111111,223.84444444444443,298.8,93.33333333333333,91.04444444444445,227.4,2.311111111111111,95.33333333333333,158.15555555555557,259.4888888888889,27.977777777777778,63.24444444444445,132.48888888888888,16.8,138.26666666666668,null,27.355555555555554,16.888888888888886,22.133333333333333,1.443555555555555,67.33333333333333,45.84444444444444,63.56222222222222,6.288888888888889,4.311111111111111,6.288888888888889,4.377777777777778,25.599999999999994,2.888888888888889,2.3555555555555556,0.4444444444444444,0.08888888888888889,13.168888888888889
stddev,null,null,1212.2664502759476,18.449220508567223,2.3851391759997758,3.1794796601159296,16.0060910123287,388.1932274244127,244.79730225621196,190.31922412280656,262.52277823249756,83.91852108931722,79.88740814294817,200.54657133126776,2.5657022981687256,90.96902569756169,150.42983195973576,225.71216809973455,22.767090372737663,70.86804497077507,135.23609359228408,13.960333416179894,113.32363149204782,null,4.65713311908613,13.464097414677275,18.433664855367205,0.7398837610043757,56.684533725138486,39.60369332843245,17.715533616381173,5.570303656739265,4.2363267870248915,6.062636023674484,4.119183992938016,25.315948979393855,2.765224158448035,2.3851391759997758,0.8133979109981547,0.28779903201415186,25.54651570462134
min,Adrián,Alavés,45,0,0,0,0,17,6,2,8,0,1,1,0,3,1,2,0,0,1,0,4,ARG,19,0.5,0,0.0,2,0,0.0,0,0,0,0,0.0,0,0,0,0,0.0
max,Ørjan Nyland,Villarreal,3420,68,9,11,62,1176,794,664,843,305,270,702,11,318,575,776,77,270,545,48,393,UKR,37,38.0,68,4.0,194,143,100.0,20,15,23,15,100.0,9,9,3,1,100.0


CREACION DE COLUMNAS | METRICAS NUEVAS

In [0]:
gk_df = gk_df.withColumns({
    # % SUCC
    "passes_total_succ_%": F.round(((gk_df['passes_opposition_half_successful'] + gk_df['passes_own_half_successful']) / gk_df['passes_total'])*100, 2),
    "short_passes_succ_%": F.round(gk_df['passes_short_successful'] / ((gk_df['passes_short_successful'] + gk_df['passes_short_unsuccessful']))*100, 2),
    "long_passes_succ_%": F.round(gk_df['passes_long_successful'] / ((gk_df['passes_long_successful'] + gk_df['passes_long_unsuccessful']))*100, 2),
    "own_half_passes_succ_%": F.round(gk_df['passes_own_half_successful'] / ((gk_df['passes_own_half_successful'] + gk_df['passes_own_half_unsuccessful']))*100, 2),
    "opp_half_passes_succ_%": F.round(gk_df['passes_opposition_half_successful'] / ((gk_df['passes_opposition_half_successful'] + gk_df['passes_opposition_half_unsuccessful']))*100, 2),
    "passes_openplay_succ_%": F.round((gk_df['passes_open_play_successful'] / gk_df['passes_open_play'])*100, 2),
    # % OVER TOTAl
    "short_passes_%": F.round(((gk_df['passes_short_successful']+gk_df['passes_short_unsuccessful'])/gk_df['passes_total'])*100, 2),
    "long_passes_%": F.round(((gk_df['passes_long_successful']+gk_df['passes_long_unsuccessful'])/gk_df['passes_total'])*100, 2),
    "own_half_passes_%": F.round(((gk_df['passes_own_half_successful']+gk_df['passes_own_half_unsuccessful'])/gk_df['passes_total'])*100, 2),
    "opp_half_passes_%": F.round(((gk_df['passes_opposition_half_successful']+gk_df['passes_opposition_half_unsuccessful'])/gk_df['passes_total'])*100, 2),
    "goals_conceded_inside_box_%": F.round(F.try_divide(gk_df['goals_conceded_inside_box'], gk_df['goals_conceded'])*100, 2),
    "goals_conceded_outside_box_%": F.round(F.try_divide(gk_df['goals_conceded_outside_box'], gk_df['goals_conceded'])*100, 2),
    "passes_openplay_%": F.round((gk_df['passes_open_play'] / gk_df['passes_total'])*100, 2),
    # x90
    "passes_x90": F.round(gk_df['passes_total']/gk_df['90s'], 2),
    "SoTA_x90": F.round(gk_df['SoTA']/gk_df['90s'], 2),
    "Saves_x90": F.round(gk_df['Saves']/gk_df['90s'], 2),
    "GA_inside_box_x90": F.round(gk_df['goals_conceded_inside_box']/gk_df['90s'], 2),
    "GA_outside_box_x90": F.round(gk_df['goals_conceded_outside_box']/gk_df['90s'], 2),
    "passes_openplay_x90": F.round(gk_df['passes_open_play_successful']/gk_df['90s'], 2),
    # OTHERS
    "goals_conceded_exclude_pens": gk_df['goals_conceded'] - gk_df['goals_conceded_penalty'],
    "Pts": gk_df['W']*3 + gk_df['D'], 
    
    })

In [0]:
gk_df = gk_df.withColumns({
    "Pts_per_90": F.round(gk_df['Pts']/gk_df['90s'], 2),
    #"Pts_per_Save": F.round(gk_df['Pts']/gk_df['Saves'], 2),
    "Pts_per_Save": F.round(F.try_divide(gk_df['Pts'], gk_df['Saves']), 2),
    "CS_per_90s": F.round(gk_df['CS']/gk_df['90s'], 2),
    "GA_per_SoTA": F.round(F.try_divide(gk_df['GA'], gk_df['SoTA']), 2),
    })


In [0]:
#columna calculo CS% & SoTA para comparar GKs BIAS de minutos jugados
gk_df = gk_df.withColumn("Diff_Adj_CS", gk_df['CS%'] * F.log1p(1 + gk_df['SoTA_x90']))

In [0]:
gk_df.select("player_name", "GA", "SoTA", "Saves", "GA90", "SoTA_x90", "Saves_x90", "Save%").display()

player_name,GA,SoTA,Saves,GA90,SoTA_x90,Saves_x90,Save%
Augusto Batalla,39,125,93,1.22,3.91,2.91,68.8
Vicente Guaita,48,129,81,1.41,3.79,2.38,62.8
Pau López,2,3,1,4.0,6.0,2.0,33.3
Rui Silva,18,55,37,1.2,3.67,2.47,67.3
André Ferreira,20,40,20,3.33,6.67,3.33,50.0
Álex Padilla,6,18,12,1.68,5.0,3.33,66.7
Marko Dmitrovic,49,145,102,1.53,4.53,3.19,66.2
Unai Marrero,3,11,8,1.5,5.5,4.0,72.7
Marc-André ter Stegen,9,23,14,1.2,3.07,1.87,60.9
Julen Agirrezabala,10,34,24,0.75,2.54,1.79,70.6


In [0]:
gk_df.columns

['player_name',
 'team',
 'played_time',
 'goals_conceded',
 'goals_conceded_penalty',
 'goals_conceded_outside_box',
 'goals_conceded_inside_box',
 'passes_total',
 'passes_open_play',
 'passes_open_play_successful',
 'passes_forward',
 'passes_leftside',
 'passes_rightside',
 'passes_short_successful',
 'passes_short_unsuccessful',
 'passes_long_successful',
 'passes_long_unsuccessful',
 'passes_own_half_successful',
 'passes_own_half_unsuccessful',
 'passes_opposition_half_successful',
 'passes_opposition_half_unsuccessful',
 'clearances',
 'recoveries',
 'Nation',
 'Age',
 '90s',
 'GA',
 'GA90',
 'SoTA',
 'Saves',
 'Save%',
 'W',
 'D',
 'L',
 'CS',
 'CS%',
 'PKatt',
 'PK_Allowed',
 'PK_Saved',
 'PK_missed',
 'PK_Save%',
 'passes_total_succ_%',
 'short_passes_succ_%',
 'long_passes_succ_%',
 'own_half_passes_succ_%',
 'opp_half_passes_succ_%',
 'passes_openplay_succ_%',
 'short_passes_%',
 'long_passes_%',
 'own_half_passes_%',
 'opp_half_passes_%',
 'goals_conceded_inside_box_%',
 

ANALIZANDO PORTEROS DE LA LIGA POR EDAD

In [0]:
gk_df.select('Age').groupBy("Age").count().display()

Age,count
31,3
27,5
34,1
37,2
29,2
20,1
32,4
22,2
35,1
23,6


In [0]:
gk_df.describe('Age').display()

summary,Age
count,45
mean,27.355555555555554
stddev,4.65713311908613
min,19
max,37


In [0]:
#TOP 15 porteros ordenando por 90s & edad
gk_df.select("player_name", "Age", "90s", "Pts_per_90").orderBy(F.desc("90s"), F.desc("Age")).limit(15).show()

+--------------------+---+----+----------+
|         player_name|Age| 90s|Pts_per_90|
+--------------------+---+----+----------+
|         David Soria| 31|38.0|      1.11|
|         Joan García| 23|38.0|      1.11|
|      Sergio Herrera| 31|37.0|      1.38|
|           Jan Oblak| 31|36.0|      1.94|
|         Álex Remiro| 29|36.0|      1.28|
|     Paulo Gazzaniga| 32|35.5|      1.15|
|      Vicente Guaita| 37|34.0|      1.44|
|Giorgi Mamardashvili| 23|34.0|      1.29|
|     Marko Dmitrovic| 32|32.0|      1.09|
|     Augusto Batalla| 28|32.0|      1.31|
|      Antonio Sivera| 27|31.2|      1.19|
|       Dominik Greif| 27|31.0|      1.32|
|           Karl Hein| 22|31.0|      0.52|
|    Thibaut Courtois| 32|30.0|       2.2|
|        Ørjan Nyland| 33|29.7|      1.14|
+--------------------+---+----+----------+



ANALISIS DEL DATASET

In [0]:
#BOX & WHISKERS DE LAS METRICAS X90

gk_df.display()

player_name,team,played_time,goals_conceded,goals_conceded_penalty,goals_conceded_outside_box,goals_conceded_inside_box,passes_total,passes_open_play,passes_open_play_successful,passes_forward,passes_leftside,passes_rightside,passes_short_successful,passes_short_unsuccessful,passes_long_successful,passes_long_unsuccessful,passes_own_half_successful,passes_own_half_unsuccessful,passes_opposition_half_successful,passes_opposition_half_unsuccessful,clearances,recoveries,Nation,Age,90s,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt,PK_Allowed,PK_Saved,PK_missed,PK_Save%,passes_total_succ_%,short_passes_succ_%,long_passes_succ_%,own_half_passes_succ_%,opp_half_passes_succ_%,passes_openplay_succ_%,short_passes_%,long_passes_%,own_half_passes_%,opp_half_passes_%,goals_conceded_inside_box_%,goals_conceded_outside_box_%,passes_openplay_%,passes_x90,SoTA_x90,Saves_x90,GA_inside_box_x90,GA_outside_box_x90,passes_openplay_x90,goals_conceded_exclude_pens,Pts,Pts_per_90,Pts_per_Save,CS_per_90s,GA_per_SoTA,Diff_Adj_CS
Vicente Guaita,Celta de Vigo,3060,48,3,4,44,775,461,385,374,227,174,508,5,96,166,562,49,42,122,42,312,ESP,37,34.0,48,1.41,129,81,62.8,14,7,13,8,23.5,3,3,0,0,0.0,77.94,99.03,36.64,91.98,25.61,83.51,66.19,33.81,78.84,21.16,91.67,8.33,59.48,22.79,3.79,2.38,1.29,0.12,11.32,45,49,1.44,0.6,0.24,0.37,41.26910885224524
Augusto Batalla,Rayo Vallecano,2880,39,6,8,31,1044,676,486,681,177,185,447,5,212,380,543,53,116,332,26,286,ARG,28,32.0,39,1.22,125,93,68.8,10,12,10,8,25.0,8,6,2,0,25.0,63.12,98.89,35.81,91.11,25.89,71.89,43.3,56.7,57.09,42.91,79.49,20.51,64.75,32.63,3.91,2.91,0.97,0.25,15.19,33,42,1.31,0.45,0.25,0.31,44.416145785450176
Rui Silva,Real Betis,1350,18,3,0,18,452,320,264,193,117,142,285,3,57,107,312,23,30,87,8,129,POR,30,15.0,18,1.2,55,37,67.3,5,5,5,4,26.7,3,3,0,0,0.0,75.66,98.96,34.76,93.13,25.64,82.5,63.72,36.28,74.12,25.88,100.0,0.0,70.8,30.13,3.67,2.47,1.2,0.0,17.6,15,20,1.33,0.54,0.27,0.33,46.32954944364894
Pau López,Girona,45,2,0,0,2,22,13,8,15,0,7,12,0,4,6,12,1,4,5,1,5,ESP,29,0.5,2,4.0,3,1,33.3,0,0,0,0,0.0,0,0,0,0,0.0,72.73,100.0,40.0,92.31,44.44,61.54,54.55,45.45,59.09,40.91,100.0,0.0,59.09,44.0,6.0,2.0,4.0,0.0,16.0,2,0,0.0,0.0,0.0,0.67,0.0
Álex Padilla,Athletic Club,320,6,1,2,4,105,57,33,87,8,10,23,0,29,53,27,8,25,45,3,34,MEX,20,3.6,6,1.68,18,12,66.7,0,1,2,0,0.0,1,1,0,0,0.0,49.52,100.0,35.37,77.14,35.71,57.89,21.9,78.1,33.33,66.67,66.67,33.33,54.29,29.17,5.0,3.33,1.11,0.56,9.17,5,1,0.28,0.08,0.0,0.33,0.0
André Ferreira,Real Valladolid,540,20,3,0,20,168,84,61,104,34,30,75,0,37,56,86,15,26,41,9,43,POR,28,6.0,20,3.33,40,20,50.0,0,0,6,0,0.0,3,3,0,0,0.0,66.67,100.0,39.78,85.15,38.81,72.62,44.64,55.36,60.12,39.88,100.0,0.0,50.0,28.0,6.67,3.33,3.33,0.0,10.17,17,0,0.0,0.0,0.0,0.5,0.0
Marc-André ter Stegen,Barcelona,675,9,1,0,9,211,172,158,96,63,52,166,2,27,16,177,7,16,11,5,59,GER,32,7.5,9,1.2,23,14,60.9,7,0,1,1,12.5,1,1,0,0,0.0,91.47,98.81,62.79,96.2,59.26,91.86,79.62,20.38,87.2,12.8,100.0,0.0,81.52,28.13,3.07,1.87,1.2,0.0,21.07,8,21,2.8,1.5,0.13,0.39,20.29176022003865
Unai Marrero,Real Sociedad,180,3,0,0,3,45,26,24,18,18,9,26,1,9,9,32,2,3,8,2,12,ESP,22,2.0,3,1.5,11,8,72.7,0,0,2,0,0.0,1,0,1,0,100.0,77.78,96.3,50.0,94.12,27.27,92.31,60.0,40.0,75.56,24.44,100.0,0.0,57.78,22.5,5.5,4.0,1.5,0.0,12.0,3,0,0.0,0.0,0.0,0.27,0.0
Marko Dmitrovic,Leganés,2880,49,7,6,43,933,405,205,825,27,81,151,0,207,575,179,30,179,545,48,312,SRB,32,32.0,49,1.53,145,102,66.2,8,11,13,10,31.3,8,7,0,1,0.0,38.37,100.0,26.47,85.65,24.72,50.62,16.18,83.82,22.4,77.6,87.76,12.24,43.41,29.16,4.53,3.19,1.34,0.19,6.41,42,35,1.09,0.34,0.31,0.34,58.731537324925036
Julen Agirrezabala,Athletic Club,1207,9,1,2,7,356,232,175,225,51,80,162,0,69,125,183,20,48,105,11,105,ESP,23,13.4,10,0.75,34,24,70.6,9,4,1,5,35.7,2,1,1,0,50.0,64.89,100.0,35.57,90.15,31.37,75.43,45.51,54.49,57.02,42.98,77.78,22.22,65.17,26.57,2.54,1.79,0.52,0.15,13.06,8,31,2.31,1.29,0.37,0.29,54.01149433030126


Databricks visualization. Run in Databricks to view.

In [0]:
gk_df.filter(F.col("90s") >= 7).display()

player_name,team,played_time,goals_conceded,goals_conceded_penalty,goals_conceded_outside_box,goals_conceded_inside_box,passes_total,passes_open_play,passes_open_play_successful,passes_forward,passes_leftside,passes_rightside,passes_short_successful,passes_short_unsuccessful,passes_long_successful,passes_long_unsuccessful,passes_own_half_successful,passes_own_half_unsuccessful,passes_opposition_half_successful,passes_opposition_half_unsuccessful,clearances,recoveries,Nation,Age,90s,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt,PK_Allowed,PK_Saved,PK_missed,PK_Save%,passes_total_succ_%,short_passes_succ_%,long_passes_succ_%,own_half_passes_succ_%,opp_half_passes_succ_%,passes_openplay_succ_%,short_passes_%,long_passes_%,own_half_passes_%,opp_half_passes_%,goals_conceded_inside_box_%,goals_conceded_outside_box_%,passes_openplay_%,passes_x90,SoTA_x90,Saves_x90,GA_inside_box_x90,GA_outside_box_x90,passes_openplay_x90,goals_conceded_exclude_pens,Pts,Pts_per_90,Pts_per_Save,CS_per_90s,GA_per_SoTA,Diff_Adj_CS
Adrián,Real Betis,1710,27,1,3,24,525,343,265,262,134,129,287,4,98,136,349,49,36,91,30,154,ESP,37,19.0,27,1.42,77,50,64.9,10,5,4,3,15.8,1,1,0,0,0.0,73.33,98.63,41.88,87.69,28.35,77.26,55.43,44.57,75.81,24.19,88.89,11.11,65.33,27.63,4.05,2.63,1.26,0.16,13.95,26,35,1.84,0.7,0.16,0.35,28.440920698275452
Julen Agirrezabala,Athletic Club,1207,9,1,2,7,356,232,175,225,51,80,162,0,69,125,183,20,48,105,11,105,ESP,23,13.4,10,0.75,34,24,70.6,9,4,1,5,35.7,2,1,1,0,50.0,64.89,100.0,35.57,90.15,31.37,75.43,45.51,54.49,57.02,42.98,77.78,22.22,65.17,26.57,2.54,1.79,0.52,0.15,13.06,8,31,2.31,1.29,0.37,0.29,54.01149433030126
Augusto Batalla,Rayo Vallecano,2880,39,6,8,31,1044,676,486,681,177,185,447,5,212,380,543,53,116,332,26,286,ARG,28,32.0,39,1.22,125,93,68.8,10,12,10,8,25.0,8,6,2,0,25.0,63.12,98.89,35.81,91.11,25.89,71.89,43.3,56.7,57.09,42.91,79.49,20.51,64.75,32.63,3.91,2.91,0.97,0.25,15.19,33,42,1.31,0.45,0.25,0.31,44.416145785450176
Jasper Cillessen,Las Palmas,2337,42,4,3,39,857,546,372,573,172,112,394,2,149,312,442,67,101,247,15,230,NED,35,25.9,42,1.62,130,88,67.7,6,8,13,3,11.1,8,4,3,1,42.9,63.36,99.49,32.32,86.84,29.02,68.13,46.21,53.79,59.39,40.61,92.86,7.14,63.71,33.09,5.02,3.4,1.51,0.12,14.36,38,26,1.0,0.3,0.12,0.32,21.631271720218688
Diego Conde,Villarreal,1963,33,3,9,24,528,371,300,244,131,153,321,8,88,111,363,40,46,79,17,175,ESP,25,21.8,33,1.51,96,63,65.6,11,6,5,2,9.1,4,3,0,1,0.0,77.46,97.57,44.22,90.07,36.8,80.86,62.31,37.69,76.33,23.67,72.73,27.27,70.27,24.22,4.4,2.89,1.1,0.41,13.76,30,39,1.79,0.62,0.09,0.34,16.8923117123272
Thibaut Courtois,Real Madrid,2700,29,3,1,28,846,662,547,381,211,254,596,7,83,160,643,42,36,125,24,236,BEL,32,30.0,29,0.97,106,77,72.6,20,6,4,11,36.7,3,3,0,0,0.0,80.26,98.84,34.16,93.87,22.36,82.63,71.28,28.72,80.97,19.03,96.55,3.45,78.25,28.2,3.53,2.57,0.93,0.03,18.23,26,66,2.2,0.86,0.37,0.27,62.763892830106734
Marko Dmitrovic,Leganés,2880,49,7,6,43,933,405,205,825,27,81,151,0,207,575,179,30,179,545,48,312,SRB,32,32.0,49,1.53,145,102,66.2,8,11,13,10,31.3,8,7,0,1,0.0,38.37,100.0,26.47,85.65,24.72,50.62,16.18,83.82,22.4,77.6,87.76,12.24,43.41,29.16,4.53,3.19,1.34,0.19,6.41,42,35,1.09,0.34,0.31,0.34,58.731537324925036
Álvaro Fernández,Sevilla,746,16,2,6,10,291,217,147,182,51,58,128,1,52,110,150,16,30,95,9,70,ESP,26,8.3,16,1.93,36,20,55.6,2,1,5,3,37.5,2,2,0,0,0.0,61.86,99.22,32.1,90.36,24.0,67.74,44.33,55.67,57.04,42.96,62.5,37.5,74.57,35.06,4.34,2.41,1.2,0.72,17.71,14,7,0.84,0.35,0.36,0.44,69.25795381684254
Joan García,Espanyol,3420,51,4,8,43,1059,610,478,695,162,202,477,7,232,343,577,48,132,302,39,393,ESP,23,38.0,51,1.34,194,143,73.7,11,9,18,8,21.1,6,4,2,0,33.3,66.95,98.55,40.35,92.32,30.41,78.36,45.7,54.3,59.02,40.98,84.31,15.69,57.6,27.87,5.11,3.76,1.13,0.21,12.58,47,42,1.11,0.29,0.21,0.26,41.38769734449966
Paulo Gazzaniga,Girona,3195,51,2,11,40,1069,794,664,551,305,213,702,6,134,227,776,44,60,189,23,247,ARG,32,35.5,51,1.44,141,90,63.8,11,8,17,8,22.2,5,2,3,0,60.0,78.2,99.15,37.12,94.63,24.1,83.63,66.23,33.77,76.7

Databricks visualization. Run in Databricks to view.

In [0]:
#GA INSIDE & OUTSIDE AREA X90 SIZE: 90s 

gk_df.select("player_name", "team", '90s', "GA90", "GA_inside_box_x90", "GA_outside_box_x90", "SoTA_x90", "Saves", "Pts_per_Save").filter(F.col("90s") >= 7).display()

player_name,team,90s,GA90,GA_inside_box_x90,GA_outside_box_x90,SoTA_x90,Saves,Pts_per_Save
Adrián,Real Betis,19.0,1.42,1.26,0.16,4.05,50,0.7
Julen Agirrezabala,Athletic Club,13.4,0.75,0.52,0.15,2.54,24,1.29
Augusto Batalla,Rayo Vallecano,32.0,1.22,0.97,0.25,3.91,93,0.45
Jasper Cillessen,Las Palmas,25.9,1.62,1.51,0.12,5.02,88,0.3
Diego Conde,Villarreal,21.8,1.51,1.1,0.41,4.4,63,0.62
Thibaut Courtois,Real Madrid,30.0,0.97,0.93,0.03,3.53,77,0.86
Marko Dmitrovic,Leganés,32.0,1.53,1.34,0.19,4.53,102,0.34
Álvaro Fernández,Sevilla,8.3,1.93,1.2,0.72,4.34,20,0.35
Joan García,Espanyol,38.0,1.34,1.13,0.21,5.11,143,0.29
Paulo Gazzaniga,Girona,35.5,1.44,1.13,0.31,3.97,90,0.46


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#LONG & SHORT PASSES % TOP 15 GKs PASSES_OPENPLAY_%     |       OWN & OPP HALF PASSES % TOP 15 GKs PASSES_OPENPLAY_%

gk_df.select("player_name", "team", "90s", "passes_x90", "passes_openplay_%", "long_passes_%", "short_passes_%", "opp_half_passes_%", "own_half_passes_%", "clearances").filter(gk_df['90s'] >= 10).orderBy(F.desc('passes_openplay_%')).limit(15).display()

player_name,team,90s,passes_x90,passes_openplay_%,long_passes_%,short_passes_%,opp_half_passes_%,own_half_passes_%,clearances
Dinko Horkas,Las Palmas,12.1,38.76,81.88,44.56,55.44,30.28,69.72,15
Wojciech Szczesny,Barcelona,15.0,26.07,79.8,20.72,79.28,11.76,88.24,23
Iñaki Peña,Barcelona,15.5,35.48,78.91,20.18,79.82,10.0,90.0,27
Thibaut Courtois,Real Madrid,30.0,28.2,78.25,28.72,71.28,19.03,80.97,24
Paulo Gazzaniga,Girona,35.5,30.11,74.28,33.77,66.23,23.29,76.71,23
Rui Silva,Real Betis,15.0,30.13,70.8,36.28,63.72,25.88,74.12,8
Diego Conde,Villarreal,21.8,24.22,70.27,37.69,62.31,23.67,76.33,17
Ørjan Nyland,Sevilla,29.7,29.87,70.24,56.48,43.52,43.07,56.93,24
Álex Remiro,Real Sociedad,36.0,32.67,66.5,46.34,53.66,32.48,67.52,33
Dominik Greif,Mallorca,31.0,29.48,65.54,57.11,42.89,45.73,54.27,28


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#SCATTER PASSES_X90 & PASSES_OPENPLAY_X90       |       PASSES_OPENPLAY_SUCC_% & PASSES_TOT_SUCC_%

gk_df.select("player_name", "team", "90s", "passes_x90", "passes_openplay_x90", "passes_total_succ_%", "passes_openplay_succ_%", "long_passes_succ_%", "short_passes_succ_%", "opp_half_passes_succ_%", "own_half_passes_succ_%").filter(gk_df['90s'] >= 6).display()

player_name,team,90s,passes_x90,passes_openplay_x90,passes_total_succ_%,passes_openplay_succ_%,long_passes_succ_%,short_passes_succ_%,opp_half_passes_succ_%,own_half_passes_succ_%
Adrián,Real Betis,19.0,27.63,13.95,73.33,77.26,41.88,98.63,28.35,87.69
Julen Agirrezabala,Athletic Club,13.4,26.57,13.06,64.89,75.43,35.57,100.0,31.37,90.15
Augusto Batalla,Rayo Vallecano,32.0,32.63,15.19,63.12,71.89,35.81,98.89,25.89,91.11
Dani Cárdenas,Rayo Vallecano,6.0,26.17,6.83,46.5,50.62,24.55,97.87,20.22,80.88
Jasper Cillessen,Las Palmas,25.9,33.09,14.36,63.36,68.13,32.32,99.49,29.02,86.84
Diego Conde,Villarreal,21.8,24.22,13.76,77.46,80.86,44.22,97.57,36.8,90.07
Thibaut Courtois,Real Madrid,30.0,28.2,18.23,80.26,82.63,34.16,98.84,22.36,93.87
Marko Dmitrovic,Leganés,32.0,29.16,6.41,38.37,50.62,26.47,100.0,24.72,85.65
Álvaro Fernández,Sevilla,8.3,35.06,17.71,61.86,67.74,32.1,99.22,24.0,90.36
André Ferreira,Real Valladolid,6.0,28.0,10.17,66.67,72.62,39.78,100.0,38.81,85.15


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#SCATTER SAVES_X90 & SAVE%  LABELS NAME & CS%

gk_df.select("player_name", "team", "90s", "CS%", "Saves_x90", "Save%", "SoTA_x90", "GA90", "GA_per_SoTA", "CS_per_90s", "recoveries").filter(gk_df['90s'] >= 6).display()

player_name,team,90s,CS%,Saves_x90,Save%,SoTA_x90,GA90,GA_per_SoTA,CS_per_90s,recoveries
Adrián,Real Betis,19.0,15.8,2.63,64.9,4.05,1.42,0.35,0.16,154
Julen Agirrezabala,Athletic Club,13.4,35.7,1.79,70.6,2.54,0.75,0.29,0.37,105
Augusto Batalla,Rayo Vallecano,32.0,25.0,2.91,68.8,3.91,1.22,0.31,0.25,286
Dani Cárdenas,Rayo Vallecano,6.0,33.3,2.33,70.0,3.33,1.0,0.3,0.33,41
Jasper Cillessen,Las Palmas,25.9,11.1,3.4,67.7,5.02,1.62,0.32,0.12,230
Diego Conde,Villarreal,21.8,9.1,2.89,65.6,4.4,1.51,0.34,0.09,175
Thibaut Courtois,Real Madrid,30.0,36.7,2.57,72.6,3.53,0.97,0.27,0.37,236
Marko Dmitrovic,Leganés,32.0,31.3,3.19,66.2,4.53,1.53,0.34,0.31,312
Álvaro Fernández,Sevilla,8.3,37.5,2.41,55.6,4.34,1.93,0.44,0.36,70
André Ferreira,Real Valladolid,6.0,0.0,3.33,50.0,6.67,3.33,0.5,0.0,43


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import percent_rank, dense_rank
from pyspark.sql import Window


#Crear dict con las métricas que queremos pasar a la funcion para crear el ranking
metrics = {
        "Saves_x90": {
            "weight": 0.25,
            "direction": "desc"
        },
        "Save%": {
            "weight": 0.25,
            "direction": "desc"
        },
        "GA90": {
            "weight": 0.10,
            "direction": "asc"   # menor es mejor
        },
        "PK_Save%": {
            "weight": 0.05,
            "direction": "desc"
        },
        "passes_openplay_succ_%": {
            "weight": 0.10,
            "direction": "desc"
        },
        "opp_half_passes_succ_%": {
            "weight": 0.05,
            "direction": "desc"
        },
        "Diff_Adj_CS": {
            "weight": 0.05,
            "direction": "desc"
        },
        "GA_per_SoTA": {
            "weight": 0.15,
            "direction": "asc"
        }

}




In [0]:
def generate_weighted_ranking_gk(df, min_90s, metrics):

    # Filtrar jugadores con suficientes minutos
    result_df = df.filter(F.col("90s") >= min_90s)

    pct_cols = []

    # Crear percentiles para cada métrica
    for metric, config in metrics.items():

        if config["direction"] == "desc":
            window = Window.orderBy(F.desc(metric))
        else:
            window = Window.orderBy(F.asc(metric))

        pct_col = f"{metric}_pct"

        result_df = result_df.withColumn(
            pct_col,
            1 - percent_rank().over(window)
        )

        pct_cols.append(pct_col)

    # Weighted score
    weighted_score = sum(
        F.col(f"{metric}_pct") * config["weight"]
        for metric, config in metrics.items()
    )

    result_df = result_df.withColumn(
        "weighted_score",
        weighted_score
    )

    # Ranking final
    result_df = result_df.withColumn(
        "Rank",
        dense_rank().over(
            Window.orderBy(F.desc("weighted_score"))
        )
    )

    # Columnas finales a mostrar
    output_cols = [
        "player_name",
        "team",
        "age",
        "90s",
        *pct_cols,
        "Rank"
    ]

    return result_df.select(*output_cols)

In [0]:
#Instanciar la funcion y generar el ranking con los parametros que queremos
gk_ranking = generate_weighted_ranking_gk(
    df=gk_df,
    min_90s=10,
    metrics = metrics
)

display(
    gk_ranking.orderBy("Rank")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


player_name,team,age,90s,Saves_x90_pct,Save%_pct,GA90_pct,PK_Save%_pct,passes_openplay_succ_%_pct,opp_half_passes_succ_%_pct,Diff_Adj_CS_pct,GA_per_SoTA_pct,Rank
Joan García,Espanyol,23,38.0,1.0,0.9166666666666666,0.41666666666666663,0.7916666666666666,0.625,0.41666666666666663,0.375,0.9583333333333334,1
Jan Oblak,Atlético de Madrid,31,36.0,0.29166666666666663,0.9583333333333334,0.875,0.5416666666666667,0.9166666666666666,0.875,0.9166666666666666,0.9583333333333334,2
Sergio Herrera,Osasuna,31,37.0,0.9583333333333334,0.7916666666666666,0.375,0.5416666666666667,0.375,1.0,0.41666666666666663,0.8333333333333334,3
Luiz Júnior,Villarreal,23,16.2,0.7916666666666666,0.8333333333333334,0.7083333333333333,0.5416666666666667,0.33333333333333337,0.625,0.75,0.8333333333333334,4
Thibaut Courtois,Real Madrid,32,30.0,0.5,0.875,0.8333333333333334,0.5416666666666667,0.7916666666666666,0.04166666666666663,0.875,0.875,5
Unai Simón,Athletic Club,27,21.0,0.08333333333333337,1.0,1.0,0.9583333333333334,0.45833333333333337,0.7083333333333333,0.9583333333333334,1.0,6
Dominik Greif,Mallorca,27,31.0,0.625,0.75,0.75,0.5416666666666667,0.20833333333333337,0.9583333333333334,0.29166666666666663,0.75,7
Augusto Batalla,Rayo Vallecano,28,32.0,0.7083333333333333,0.625,0.5416666666666667,0.7083333333333333,0.25,0.29166666666666663,0.5,0.625,8
Jasper Cillessen,Las Palmas,35,25.9,0.9166666666666666,0.5833333333333333,0.04166666666666663,0.8333333333333334,0.16666666666666663,0.375,0.08333333333333337,0.5833333333333333,9
Julen Agirrezabala,Athletic Club,23,13.4,0.04166666666666663,0.7083333333333333,0.9583333333333334,0.9583333333333334,0.41666666666666663,0.45833333333333337,0.7083333333333333,0.75,10
